# Reading Raw Data

In [0]:
df = spark.table("workspace.raw_crashes.crashes_table")

# Data Transformation

## Fix Date datatype

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

df = df.withColumn("CRASH_DATE", substring_index("CRASH_DATE", " ", 1))
df = df.withColumn("CRASH_DATE", to_date(col("CRASH_DATE"), "MM/dd/yyyy"))



## Trimming

In [0]:
#trim the string columns
# normalization 

for field in df.schema.fields:
    print(field)
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,trim(col(field.name)))
    

    


In [0]:
from pyspark.sql.functions import sum 
cols_to_keep = []
df_nulls = df.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df.columns
]).collect()[0].asDict()


for key, value in df_nulls.items():
    if value < 10000:
        cols_to_keep.append(key)
        
print(cols_to_keep)


df = df.select(cols_to_keep)




# Remove & Cleaning Nulls

In [0]:
string_nulls = ["", " ", "NA", "N/A", "null", "None"]
numeric_nulls = [0, -1, 9999]
df_clean =df

df.dtypes

for column, data_type in df.dtypes:
    if data_type == 'string':
        df_clean = df.withColumn(column, when(col(column).isNull() | (trim(col(column)).isin(string_nulls)),None) \
                                 .otherwise(col(column)))
        
    elif data_type in ("int", "bigint", "double", "float"):
        df_clean = df.withColumn(column, when(col(column).isNull() | (col(column).isin(numeric_nulls)),0) \
                                 .otherwise(col(column)))
         
        
df_clean = df_clean.fillna({"latitude": 0})
df_clean = df_clean.fillna({"longitude": 0})
injuries_cols = ["INJURIES_TOTAL", "INJURIES_FATAL", "INJURIES_INCAPACITATING", "INJURIES_NON_INCAPACITATING", "INJURIES_REPORTED_NOT_EVIDENT", "INJURIES_NO_INDICATION", "INJURIES_UNKNOWN"]

for x in injuries_cols:
    df_clean = df_clean.fillna({f"{x}": 0})






# WRITE TO CLEAN SILVER TABLE

In [0]:
df_clean.write \
    .mode('overwrite') \
    .format("delta") \
    .saveAsTable("workspace.silver_crashes.crashes_table")
        